# Incident Commander — GRPO training on free Colab T4

End-to-end notebook: clones the env repo, installs Unsloth + TRL, boots the env server as a subprocess, records the pre-training baseline, runs GRPO for N iterations on the `easy_canary_regression` task, records the post-training baseline, and (optionally) pushes the LoRA adapter to your HF Hub.

**Runtime:** `Runtime → Change runtime type → T4 GPU` (free tier). About 14 GB VRAM available; Llama-3.2-3B 4-bit comfortably fits with LoRA rank 16.

**Expected wall time:** 30–90 min for 50 iterations × 4 rollouts (episodes are ~7 steps each).

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Clone the repo

Replace the URL with your fork if you've pushed local changes (the env simulator + graders live in this repo).

In [ ]:
import os
REPO_URL = os.environ.get('OPENENV_REPO', 'https://github.com/YOUR_USER/OpenEnv.git')
!git clone {REPO_URL} /content/OpenEnv
%cd /content/OpenEnv/incident_commander

## 3. Install dependencies

Unsloth's Colab install one-liner plus our env package. Unsloth pulls in the right torch / transformers / bitsandbytes for Colab's CUDA version automatically.

In [ ]:
!pip -q install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip -q install --no-deps "trl<0.9.0" peft accelerate bitsandbytes
!pip -q install -e /content/OpenEnv/incident_commander

## 4. Hugging Face login

Paste a token with **`write`** scope (needed later to push the LoRA adapter). Reading `unsloth/Llama-3.2-3B-Instruct` is public; the token is only required for the push step.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 5. Configure the env server

These env vars are read by `server/app.py` **when the server process starts**. Change `IC_SEED` between training runs if you want a different scenario instance.

In [ ]:
%env IC_TASK_ID=easy_canary_regression
%env IC_SEED=0
%env IC_STEP_BUDGET=25
%env HOST=0.0.0.0
%env PORT=8000

## 6. Start the env server in the background

`uvicorn` stays alive for the notebook lifetime. The subprocess inherits `IC_TASK_ID` / `IC_SEED` from the `%env` cell above.

In [ ]:
import subprocess, time, urllib.request, urllib.error

server_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'server.app:app',
     '--host', '0.0.0.0', '--port', '8000', '--log-level', 'warning'],
    cwd='/content/OpenEnv/incident_commander',
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)

for _ in range(30):
    try:
        with urllib.request.urlopen('http://localhost:8000/openapi.json', timeout=1) as r:
            if r.status == 200:
                print('env server up')
                break
    except (urllib.error.URLError, TimeoutError):
        time.sleep(1)
else:
    raise RuntimeError('env server failed to start; check `server_proc.poll()`')

## 7. Run GRPO

Records pre-training baseline, trains, records post-training baseline, saves LoRA to `/content/ic-grpo-lora`.

If the T4 runs out of memory, drop `--group-size` to 2 or switch to a smaller base (`unsloth/Qwen2.5-1.5B-Instruct`).

In [ ]:
!python -m incident_commander.training.train_grpo \
  --base-model unsloth/Llama-3.2-3B-Instruct \
  --env-url http://localhost:8000 \
  --iterations 50 \
  --group-size 4 \
  --lr 5e-6 \
  --lora-rank 16 \
  --eval-episodes 3 \
  --output-dir /content/ic-grpo-lora

## 8. Push the LoRA to HF Hub (optional)

Rename the repo id to something under your account. This makes the trained adapter reproducible and citeable in the README / demo.

In [ ]:
from huggingface_hub import HfApi
HF_REPO_ID = 'YOUR_USER/incident-commander-llama3.2-3b-grpo'
api = HfApi()
api.create_repo(HF_REPO_ID, private=False, exist_ok=True)
api.upload_folder(
    folder_path='/content/ic-grpo-lora',
    repo_id=HF_REPO_ID,
    commit_message='GRPO-trained LoRA on easy_canary_regression',
)
print(f'https://huggingface.co/{HF_REPO_ID}')

## 9. Clean up

Kills the env server subprocess so a later cell running another training config starts fresh.

In [ ]:
server_proc.terminate()
server_proc.wait(timeout=5)
print('env server stopped')